# Yobareta (呼ばれた) — Evaluation & Audio Benchmark Suite
This notebook runs:
1. **Unit Tests (`pytest`)** on `backend/matcher.py` (Japanese SudachiPy morae, long-vowel normalization, English phonetic sliding matcher, surname-only cap, and room-number guards).
2. **26-Call Benchmark Verification (`eval/score.py --verify-matcher`)** across all hospital, restaurant, and café scenarios in `eval/calls.csv`.
3. **Synthetic Waiting-Room Audio Generation (`eval/generate_audio.py`)** creating 16 kHz mono PCM `.wav` clips for all 26 announcements.
4. **Interactive Audio Player & Live Matcher Inspector** so you can play any clip out loud (to test the Android app) or inspect its matcher output.

In [ ]:
# 1. Clone / pull the latest repo and install dependencies
!rm -rf /content/df-2026-t044
!git clone https://github.com/Devfest-Hackathon-2026/df-2026-t044.git /content/df-2026-t044
%cd /content/df-2026-t044
!pip install -q SudachiPy==0.6.8 SudachiDict-core==20240716 gTTS pydub pytest

In [ ]:
# 2. Run unit tests & 26-utterance benchmark verification
!pytest backend/tests/test_matcher.py
!python3 eval/score.py --verify-matcher

In [ ]:
# 3. Generate 16 kHz mono WAV audio clips for all 26 test announcements
!python3 eval/generate_audio.py

In [ ]:
# 4. Interactive Audio Player + Matcher Output for all 26 Benchmark Calls
import csv
import sys
from pathlib import Path
from IPython.display import Audio, display, HTML

sys.path.insert(0, '/content/df-2026-t044/backend')
from matcher import evaluate_transcript
from eval.score import PROFILES

with open('eval/calls.csv', encoding='utf-8-sig') as f:
    calls = list(csv.DictReader(f))

for row in calls:
    cid = row['id']
    prof = PROFILES[row['profile']]
    utt = '' if row['announcement'].startswith('(') else row['announcement']
    res = evaluate_transcript(
        transcript=utt,
        full_name_readings=prof['full_name_readings'],
        surname_readings=prof['surname_readings'],
        ticket_number=prof['ticket_number'],
        kanji_name=prof['kanji_name'],
    )
    wav_path = Path(f'eval/audio/{cid}.wav')
    display(HTML(
        f"<b>[{cid}] ({row['category']})</b> — Profile {row['profile']} ({prof['kanji_name']})<br>"
        f"<b>Announcement:</b> {row['announcement']}<br>"
        f"<b>Matcher Output:</b> tier=<code>{res.tier}</code> | score=<code>{res.score}</code> | "
        f"dest=<code>{res.destination}</code> | banner=<code>{res.instruction_plain}</code>"
    ))
    if wav_path.exists():
        display(Audio(str(wav_path), autoplay=False))
    display(HTML('<hr>'))

In [ ]:
# 5. (Optional) Run End-to-End Audio Upload Test against a Live Cloud Run Backend
# Paste your Cloud Run URL below once deployed:
CLOUD_RUN_URL = ""  # e.g. "https://yobareta-backend-696344028704.asia-northeast1.run.app"

if CLOUD_RUN_URL:
    !python3 eval/test_audio_e2e.py --url {CLOUD_RUN_URL}
else:
    print("Set CLOUD_RUN_URL above to run live Cloud Run + Chirp 3 STT audio tests.")